# Phase 1 — Creating Arrays & Understanding What They Are

NumPy is the foundation everything else in AI sits on. Pandas, scikit-learn, PyTorch and
TensorFlow all store their numbers as NumPy-style arrays under the hood. So learning NumPy
well means the next four libraries come much easier.

Everything in NumPy revolves around one object: the **`ndarray`** ("n-dimensional array").

### So what is an array, really?

Think of a **Python list** as a coat-check room. The list holds numbered tickets, and each
ticket points to a coat stored somewhere else in the building. To look at 5 coats, you walk
to 5 different rooms.

A **NumPy array** is more like an egg carton. The eggs sit side by side in one solid block.
Grab the carton and you have all of them at once.

That difference is the whole story:

| Python list (coat check) | NumPy array (egg carton) |
|---|---|
| Holds pointers to values stored elsewhere | Holds the raw numbers side by side in one block |
| Can mix types — numbers, text, other lists | Every slot is the same type (all numbers, say) |
| Looping happens in slow Python | Looping happens in fast, pre-compiled C code |
| Can grow and shrink freely | Fixed size — changing the size makes a new array |

Because the numbers sit next to each other, your CPU can grab a whole run of them in one
trip instead of chasing pointers around memory. That is where the speed comes from.

### What this notebook covers
1. Why bother? — an actual speed measurement
2. Making arrays out of Python lists
3. Making arrays from scratch (zeros, ones, ranges, random)
4. Reading an array's "vital statistics" (shape, size, type)
5. Data types, and two surprises they cause
6. Changing an array's shape
7. Views vs copies — the #1 thing that bites beginners

In [1]:
# ==========================================
# SETUP — this is the only import we need
# ==========================================
import numpy as np          # "np" is the universal nickname for numpy. Everyone uses it.

print("NumPy version:", np.__version__)

NumPy version: 2.4.4


## 1. Why bother with NumPy? Let's measure it

Talk is cheap. Below we square one million numbers twice — once the plain Python way, once
the NumPy way — and time both.

In [2]:
# ==========================================
# 1. SPEED TEST — python list vs numpy array
# ==========================================
import time

size = 1_000_000                 # one million numbers
py_list = list(range(size))      # the plain Python version
np_arr  = np.arange(size)        # the NumPy version of the same numbers

# --- Step A: square every number the Python way (a real loop, one number at a time) ---
start = time.perf_counter()                    # start the stopwatch
py_result = [x * x for x in py_list]           # Python visits all 1,000,000 items itself
py_time = time.perf_counter() - start          # stop the stopwatch

# --- Step B: square every number the NumPy way (no loop written at all) ---
start = time.perf_counter()
np_result = np_arr ** 2                        # one instruction handles all 1,000,000
np_time = time.perf_counter() - start

print(f"Python list : {py_time:.4f} seconds")
print(f"NumPy array : {np_time:.4f} seconds")
print(f"NumPy is about {py_time / np_time:.0f}x faster")

# --- Step C: NumPy also uses far less memory ---
import sys
int_obj  = sys.getsizeof(py_list[-1])              # every Python number is its own object
list_mem = sys.getsizeof(py_list) + size * int_obj # the tickets PLUS all the coats
print(f"\nlist  : {list_mem / 1e6:6.1f} MB  "
      f"({sys.getsizeof(py_list)/1e6:.0f} MB of pointers + {int_obj} bytes per number)")
print(f"array : {np_arr.nbytes / 1e6:6.1f} MB  (just the raw numbers, nothing wrapped)")

Python list : 0.0181 seconds
NumPy array : 0.0015 seconds
NumPy is about 12x faster

list  :   36.0 MB  (8 MB of pointers + 28 bytes per number)
array :    8.0 MB  (just the raw numbers, nothing wrapped)


> ### The one idea to take from this: **vectorization**
>
> "Vectorized" is jargon for something simple: *you write one instruction that applies to
> the whole array, instead of writing a loop.* `np_arr ** 2` says "square everything" and
> NumPy runs that loop for you down in fast C code.
>
> Practical rule: **whenever you catch yourself writing a `for` loop over an array, stop and
> look for the array version.** There is almost always one, and it's almost always faster
> and shorter.

## 2. Making arrays out of Python lists

`np.array()` takes a list and turns it into an array. How many lists you nest inside each
other decides the number of **dimensions** — NumPy calls this `ndim`.

The three you'll meet constantly:
- **1 dimension** — a plain row of numbers. Often called a *vector*.
- **2 dimensions** — rows and columns, like a spreadsheet. Called a *matrix*.
- **3+ dimensions** — a stack of spreadsheets. Called a *tensor* (yes, as in TensorFlow).

In [3]:
# ==========================================
# 2. BUILDING ARRAYS FROM PYTHON LISTS
# ==========================================

# 1-D: one flat row of numbers, like a shopping list
vector = np.array([1, 2, 3, 4])

# 2-D: a list OF lists. Each inner list becomes one row, like a spreadsheet
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])

# 3-D: a list of 2-D blocks — think two spreadsheets stacked on top of each other
tensor = np.array([[[1, 2], [3, 4]],
                   [[5, 6], [7, 8]]])

print("vector:", vector, "| ndim =", vector.ndim, "| shape =", vector.shape)
print("\nmatrix:\n", matrix, "\nndim =", matrix.ndim, "| shape =", matrix.shape)
print("\ntensor:\n", tensor, "\nndim =", tensor.ndim, "| shape =", tensor.shape)

vector: [1 2 3 4] | ndim = 1 | shape = (4,)

matrix:
 [[1 2 3]
 [4 5 6]
 [7 8 9]] 
ndim = 2 | shape = (3, 3)

tensor:
 [[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]] 
ndim = 3 | shape = (2, 2, 2)


### How to read a "shape"

`shape` is just a list of sizes, counted from the outside in. Read it out loud:

- `(4,)` → "4 items". The lonely comma is Python's way of writing a one-item tuple.
- `(3, 3)` → "3 rows, each 3 long".
- `(2, 2, 2)` → "2 blocks, each 2 rows, each row 2 long".

Later in AI work you'll see shapes like `(32, 224, 224, 3)`. Nothing new is going on — that
just reads as "32 photos, each 224 pixels tall, 224 pixels wide, 3 colour values per pixel".

⚠️ **One rule when building from lists: every row must be the same length.** An array is an
egg carton, and egg cartons have equal-sized slots. Uneven lists have nowhere to go.

In [4]:
# What happens if the rows are different lengths? NumPy refuses.
try:
    bad = np.array([[1, 2, 3],
                    [4, 5]])          # 3 items in row one, only 2 in row two
except ValueError as e:
    print("NumPy said no:", e)

NumPy said no: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


## 3. Making arrays from scratch

Usually you don't have the numbers yet — you just want an array of the right *size* to fill
in later, the way you'd rule up an empty table before writing in it.

One thing that trips everybody up: **the size goes in as a tuple** (a pair of numbers in
their own brackets). It's `np.zeros((3, 4))`, not `np.zeros(3, 4)`.

In [5]:
# ==========================================
# 3a. ARRAYS FILLED WITH A FIXED VALUE
# ==========================================

# All zeros — the usual starting point for a counter or an empty result table
print("zeros, 3 rows x 4 cols:\n", np.zeros((3, 4)))

# All ones — handy as a starting point, or as a "keep everything" mask
print("\nones, 2 rows x 3 cols:\n", np.ones((2, 3)))

# All the same value of your choosing
print("\nfull of 7s:\n", np.full((2, 3), 7))

# The identity matrix: 1s on the diagonal, 0s everywhere else.
# In matrix maths this behaves like the number 1 does in normal maths.
print("\nidentity, 3x3:\n", np.eye(3))

zeros, 3 rows x 4 cols:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

ones, 2 rows x 3 cols:
 [[1. 1. 1.]
 [1. 1. 1.]]

full of 7s:
 [[7 7 7]
 [7 7 7]]

identity, 3x3:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [6]:
# ==========================================
# 3b. ARRAYS THAT COUNT
# ==========================================

# arange = "array range". Like Python's range(), but you get an array back.
# Reads as: start at 0, stop before 10, take steps of 2
print("arange(0, 10, 2)  :", np.arange(0, 10, 2))

# linspace = "linearly spaced". You say HOW MANY numbers you want, not how big the gap is.
# Reads as: give me 5 evenly spread numbers from 0 to 1, including both ends
print("linspace(0, 1, 5) :", np.linspace(0, 1, 5))

# empty = grab some memory but don't bother clearing it out first.
# It's the fastest option, but whatever junk was in that memory is what you get.
# Never read from it before you've written to it.
print("\nempty(3) — leftover memory, don't trust it:", np.empty(3))

# The "_like" family: copy the shape of an array you already have
template = np.array([[1, 2, 3],
                     [4, 5, 6]])
print("\nzeros shaped like the template:\n", np.zeros_like(template))

arange(0, 10, 2)  : [0 2 4 6 8]
linspace(0, 1, 5) : [0.   0.25 0.5  0.75 1.  ]

empty(3) — leftover memory, don't trust it: [1. 1. 1.]

zeros shaped like the template:
 [[0 0 0]
 [0 0 0]]


### Random numbers — and why "random" here isn't really random

Computers can't produce true randomness. They produce a fixed sequence that *looks* random,
starting from a number called the **seed**. Same seed → same "random" numbers every time.

That sounds like a flaw. It's actually the feature you want: it means your notebook gives
the same answers tomorrow, and someone else running your code sees what you saw.

**Two ways to do this, and it matters which you pick:**
- **The modern way** — `rng = np.random.default_rng(42)` hands you your own private
  random-number machine. Nothing else can interfere with it.
- **The old way** — `np.random.seed(42)` sets one *global* seed shared by your whole
  program, so any library you import can reseed it and quietly change your results.

Use the modern way in new code. The old way appears below only because you'll meet it in
almost every tutorial and older project.

In [7]:
# ==========================================
# 3c. RANDOM NUMBERS
# ==========================================

# Create your own random-number machine. Seed 42 makes it repeat the same sequence.
rng = np.random.default_rng(seed=42)

# Decimals between 0 and 1
print("random decimals:\n", rng.random((2, 3)).round(3))

# Whole numbers — low is included, high is NOT, so this gives 1 to 6 like a dice
print("\ndice rolls    :", rng.integers(1, 7, size=8))

# The bell curve: most values land near 'loc', spread controlled by 'scale'
print("\nbell curve    :", rng.normal(loc=0, scale=1, size=5).round(3))

# The older style you'll see everywhere else:
np.random.seed(42)
print("\nold style     :\n", np.random.random((2, 3)).round(3))

random decimals:
 [[0.774 0.439 0.859]
 [0.697 0.094 0.976]]

dice rolls    : [5 5 5 5 4 1 6 3]

bell curve    : [0.879 0.778 0.066 1.127 0.468]

old style     :
 [[0.375 0.951 0.732]
 [0.599 0.156 0.156]]


## 4. An array's vital statistics

Every array knows a few things about itself. Learn these five — when NumPy throws an error
at you, **printing `.shape` and `.dtype` explains what went wrong about 80% of the time.**

In [8]:
# ==========================================
# 4. INSPECTING AN ARRAY
# ==========================================
arr = np.array([[1, 2, 3, 4, 5],
                [6, 7, 8, 9, 10]])

print("the array:\n", arr)
print("\nshape    :", arr.shape,    "→ 2 rows, 5 columns")
print("ndim     :", arr.ndim,     "→ how many dimensions (2 = a table)")
print("size     :", arr.size,     "→ total number of items (2 x 5)")
print("dtype    :", arr.dtype,    "→ what kind of number is in every slot")
print("itemsize :", arr.itemsize, "→ bytes taken by ONE number")
print("nbytes   :", arr.nbytes,   "→ bytes taken by the whole array (size x itemsize)")

the array:
 [[ 1  2  3  4  5]
 [ 6  7  8  9 10]]

shape    : (2, 5) → 2 rows, 5 columns
ndim     : 2 → how many dimensions (2 = a table)
size     : 10 → total number of items (2 x 5)
dtype    : int64 → what kind of number is in every slot
itemsize : 8 → bytes taken by ONE number
nbytes   : 80 → bytes taken by the whole array (size x itemsize)


## 5. Data types — and two surprises

An array holds **one type of thing only**. That's the trade you make for the speed: the
egg carton has one size of slot.

`dtype` ("data type") is the name of that type. `int64` means "whole number using 64 bits";
`float64` means "decimal number using 64 bits".

If you hand NumPy a mixture, it quietly picks one type that can hold everything — and its
choice isn't always the one you wanted.

In [9]:
# ==========================================
# 5a. WHAT TYPE DOES NUMPY PICK?
# ==========================================
print("all whole numbers :", np.array([1, 2, 3]).dtype)
print("all decimals      :", np.array([1.0, 2.0]).dtype)
print("mixed together    :", np.array([1, 2.5]).dtype,
      "  ← the whole number became a decimal, so nothing is lost")
print("a string in there :", np.array([1, "two"]).dtype,
      "  ← EVERYTHING became text! You can't do maths on this any more")

# You can demand a specific type. Smaller type = less memory.
small = np.array([1, 2, 3], dtype=np.int8)     # int8 = 1 byte each instead of 8
print("\nint8 array:", small, "| total bytes =", small.nbytes)

all whole numbers : int64
all decimals      : float64
mixed together    : float64   ← the whole number became a decimal, so nothing is lost
a string in there : <U21   ← EVERYTHING became text! You can't do maths on this any more

int8 array: [1 2 3] | total bytes = 3


In [10]:
# ==========================================
# 5b. TWO SURPRISES THAT CAUSE REAL BUGS
# ==========================================

# SURPRISE 1: small number types wrap around like a car odometer.
# int8 can only count up to 127. Add 1 and it silently rolls over to -128.
# No error, no warning — just a wrong number.
odometer = np.array([127], dtype=np.int8)
print("int8 maxes out at 127.  127 + 1 =", odometer + 1, "← it wrapped around!")

# SURPRISE 2: astype(int) CHOPS the decimal off. It does not round.
# 2.5 becomes 2, and -3.7 becomes -3 (chopped toward zero, not down).
prices = np.array([1.9, 2.5, -3.7])
print("\nastype(int)      :", prices.astype(int), "  ← decimals chopped off")
print("np.round then int:", np.round(prices).astype(int), "  ← properly rounded")

int8 maxes out at 127.  127 + 1 = [-128] ← it wrapped around!

astype(int)      : [ 1  2 -3]   ← decimals chopped off
np.round then int: [ 2  2 -4]   ← properly rounded


## 6. Changing an array's shape

Reshaping doesn't move any numbers around — it just changes how NumPy *reads* the same
block of memory. Same 12 eggs, rearranged from one long tray into 3 rows of 4.

**The only rule:** the total count has to stay the same. 12 items can become 3×4 or 2×6 or
12×1, but never 5×3.

In [11]:
# ==========================================
# 6a. RESHAPING
# ==========================================
arr = np.arange(12)                  # the numbers 0 to 11 in one long row
print("we start with 12 numbers:", arr)

print("\nas 3 rows of 4:\n", arr.reshape(3, 4))
print("\nas 2 rows of 6:\n", arr.reshape(2, 6))

# -1 is a shortcut meaning "you figure this one out".
# We say "3 rows", NumPy works out that each must be 4 long.
print("\nreshape(3, -1) — NumPy fills in the 4:\n", arr.reshape(3, -1))

# Ask for a shape that doesn't add up and you get a clear error
try:
    arr.reshape(5, 3)                # 5 x 3 = 15, but we only have 12 numbers
except ValueError as e:
    print("\nDoesn't add up:", e)

we start with 12 numbers: [ 0  1  2  3  4  5  6  7  8  9 10 11]

as 3 rows of 4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

as 2 rows of 6:
 [[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]

reshape(3, -1) — NumPy fills in the 4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Doesn't add up: cannot reshape array of size 12 into shape (5,3)


### `flatten()` vs `ravel()` — two ways to squash an array flat

Both turn any array into one long row. The difference is what you get back:

- **`flatten()`** hands you a **photocopy**. Scribble on it and the original is untouched.
- **`ravel()`** hands you a **window onto the original**. Scribble on it and you've scribbled
  on the original.

`ravel()` is faster because it doesn't copy anything. But if you don't know which one you're
holding, you can corrupt your data without a single error message. Let's watch it happen.

In [12]:
# ==========================================
# 6b. PHOTOCOPY vs WINDOW
# ==========================================
matrix = np.arange(6).reshape(2, 3)
print("original matrix:\n", matrix)

photocopy = matrix.flatten()     # an independent copy
window    = matrix.ravel()       # a window looking at the same numbers

print("\nflatten() gives:", photocopy, "| is it independent?", photocopy.base is None)
print("ravel()   gives:", window,    "| is it independent?", window.base is None)

# --- Scribble on the photocopy: the original is safe ---
photocopy[0] = 999
print("\nafter changing the flatten() result, the matrix is fine:\n", matrix)

# --- Scribble on the window: the original changed too! ---
window[0] = 999
print("\nafter changing the ravel() result, the matrix CHANGED:\n", matrix)

original matrix:
 [[0 1 2]
 [3 4 5]]

flatten() gives: [0 1 2 3 4 5] | is it independent? True
ravel()   gives: [0 1 2 3 4 5] | is it independent? False

after changing the flatten() result, the matrix is fine:
 [[0 1 2]
 [3 4 5]]

after changing the ravel() result, the matrix CHANGED:
 [[999   1   2]
 [  3   4   5]]


In [13]:
# ==========================================
# 6c. FLIPPING AND ADDING DIMENSIONS
# ==========================================
arr = np.arange(6).reshape(2, 3)

# Transpose (.T) flips rows into columns, like turning a table on its side
print("before, 2 rows x 3 cols:\n", arr)
print("\nafter .T, 3 rows x 2 cols:\n", arr.T)

# np.newaxis adds an empty dimension. This looks pointless now, but it's how you
# line up shapes so two arrays can be combined (Phase 5 covers why).
v = np.array([1, 2, 3])
print("\nplain row, shape        :", v.shape)
print("v[:, np.newaxis], shape :", v[:, np.newaxis].shape, "→ standing up as a column")
print("v[np.newaxis, :], shape :", v[np.newaxis, :].shape, "→ lying down as a row")
print("\nthe column version:\n", v[:, np.newaxis])

before, 2 rows x 3 cols:
 [[0 1 2]
 [3 4 5]]

after .T, 3 rows x 2 cols:
 [[0 3]
 [1 4]
 [2 5]]

plain row, shape        : (3,)
v[:, np.newaxis], shape : (3, 1) → standing up as a column
v[np.newaxis, :], shape : (1, 3) → lying down as a row

the column version:
 [[1]
 [2]
 [3]]


## 7. Views vs copies — the #1 thing that bites beginners

We saw this with `ravel` above. It's worth its own section because it explains a whole
category of "but I never changed that variable!" bugs.

- A **view** is a window onto the original array's memory. Two names, one set of numbers.
  Change either name and both appear to change, because there's really only one array.
- A **copy** is a separate array with its own memory. Totally independent.

NumPy chooses for you, and the rule isn't obvious, so here it is:

| What you write | What you get |
|---|---|
| `a[1:3]` — slicing with a colon | **view** (window) |
| `a.reshape(...)`, `a.T`, `a.ravel()` | **view** (window) |
| `a[[0, 2]]` — indexing with a list | **copy** |
| `a[a > 2]` — filtering by a condition | **copy** |
| `a.copy()`, `a.flatten()` | **copy** |

**The habit that saves you:** if you slice something and plan to *change* it, add `.copy()`.
The cost is tiny, and it removes an entire class of bug.

In [14]:
# ==========================================
# 7. WINDOWS vs COPIES, SIDE BY SIDE
# ==========================================

# --- Without .copy(): editing the slice edits the original ---
original = np.array([1, 2, 3, 4, 5])
piece = original[1:4]           # a WINDOW onto items 1, 2, 3
piece[0] = 99                   # we only meant to change 'piece'...
print("no .copy()   → original became:", original, "← surprise, it changed")

# --- With .copy(): the original is protected ---
original = np.array([1, 2, 3, 4, 5])
piece = original[1:4].copy()    # a genuine, separate COPY
piece[0] = 99
print("with .copy() → original stayed:", original, "← safe")

# --- How to check which one you're holding ---
# .base points back at the array you're a window onto. It's None for a real copy.
a = np.arange(5)
print("\nis a[1:4] a window onto a?    ", a[1:4].base is a)
print("is a[[1,2,3]] its own copy?   ", a[[1, 2, 3]].base is None)

no .copy()   → original became: [ 1 99  3  4  5] ← surprise, it changed
with .copy() → original stayed: [1 2 3 4 5] ← safe

is a[1:4] a window onto a?     True
is a[[1,2,3]] its own copy?    True


## Cheat sheet

| What you want | What to type |
|---|---|
| Turn a Python list into an array | `np.array([...])` |
| An array pre-filled with a value | `np.zeros`, `np.ones`, `np.full`, `np.eye` |
| A counting sequence | `np.arange(start, stop, step)` or `np.linspace(start, stop, count)` |
| Random numbers | `rng = np.random.default_rng(42)`, then `rng.random`, `rng.integers`, `rng.normal` |
| Check what you've got | `.shape`, `.ndim`, `.size`, `.dtype`, `.nbytes` |
| Rearrange it | `.reshape(...)`, `.ravel()`, `.flatten()`, `.T`, `[:, np.newaxis]` |
| Make sure it's independent | `.copy()` |

### The three things worth memorising

1. **Don't write loops over arrays.** Write one expression that covers the whole array.
2. **When something breaks, print `.shape` and `.dtype` first.** That's usually the answer.
3. **Slices are windows, not copies.** Add `.copy()` before you edit one.

### Practice

1. Build a 5×5 array with 1s all around the border and 0s in the middle.
   *(Hint: start with `np.ones`, then set the inside to 0 with a slice.)*
2. Turn `np.arange(24)` into a `(2, 3, 4)` array and print the item at `[1, 2, 3]`.
3. Make a 3×3 grid of dice rolls with a seeded generator. Run the cell twice — do you get
   the same numbers? Why?
4. Set `a = np.arange(10)` and `b = a[::2]`, then run `b[0] = 100`.
   **Predict what `a` looks like before you print it.**

In [15]:
# ==========================================
# PRACTICE 1 — worked solution
# ==========================================
border = np.ones((5, 5), dtype=int)   # Step A: start with every slot set to 1
border[1:-1, 1:-1] = 0                # Step B: hollow out the middle
                                      #   1:-1 means "skip the first and last" — on both axes
print(border)

[[1 1 1 1 1]
 [1 0 0 0 1]
 [1 0 0 0 1]
 [1 0 0 0 1]
 [1 1 1 1 1]]
